# Model Comparison Dashboard

This notebook brings together the outputs from **05_manual_linear_regression.ipynb**, **06_sklearn_regression_models.ipynb**, and **07_ensemble_learning.ipynb** into one polished comparison view.

It does four things:
1. Loads the saved models from `../models/`
2. Evaluates every model on the same validation dataset
3. Builds a clean comparison table and visual dashboard
4. Saves report-ready CSV files and PNG figures into `../reports/results/` and `../reports/figures/`


In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', context='talk')

reports_dir = Path('../reports')
figures_dir = reports_dir / 'figures'
results_dir = reports_dir / 'results'
figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

models_dir = Path('../models')
X_val = pd.read_csv('../data/processed/X_valid_scaled.csv')
y_val = pd.read_csv('../data/processed/y_valid.csv').values.ravel()

family_palette = {
    'Manual': '#E76F51',
    'Classical ML': '#2A9D8F',
    'Ensemble': '#264653'
}

print(f'Validation feature shape: {X_val.shape}')
print(f'Validation target shape: {y_val.shape}')
print(f'Figures will be saved to: {figures_dir.resolve()}')
print(f'Results will be saved to: {results_dir.resolve()}')

## Evaluate Saved Models

In [ ]:
def load_pickle_model(model_path):
    with open(model_path, 'rb') as f:
        return pickle.load(f)


def predict_manual_linear_regression(model_bundle, X_frame):
    feature_columns = model_bundle['feature_columns']
    X_manual = X_frame.reindex(columns=feature_columns, fill_value=0)
    weights = np.asarray(model_bundle['weights']).reshape(-1)
    bias_array = np.asarray(model_bundle['bias']).reshape(-1)
    bias = float(bias_array[0]) if bias_array.size else float(model_bundle['bias'])
    return X_manual.to_numpy() @ weights + bias


model_specs = [
    {'name': 'Manual Linear Regression', 'family': 'Manual', 'file': 'manual_linear_regression.pkl', 'manual': True},
    {'name': 'KNN', 'family': 'Classical ML', 'file': 'knn.pkl', 'manual': False},
    {'name': 'Linear Regression', 'family': 'Classical ML', 'file': 'linear_regression.pkl', 'manual': False},
    {'name': 'Decision Tree', 'family': 'Classical ML', 'file': 'decision_tree.pkl', 'manual': False},
    {'name': 'SVM', 'family': 'Classical ML', 'file': 'svm.pkl', 'manual': False},
    {'name': 'Random Forest', 'family': 'Ensemble', 'file': 'random_forest.pkl', 'manual': False},
    {'name': 'AdaBoost', 'family': 'Ensemble', 'file': 'boosting_model.pkl', 'manual': False},
    {'name': 'Bagging', 'family': 'Ensemble', 'file': 'bagging_model.pkl', 'manual': False},
    {'name': 'Stacking', 'family': 'Ensemble', 'file': 'stacking_model.pkl', 'manual': False},
    {'name': 'Voting', 'family': 'Ensemble', 'file': 'voting_model.pkl', 'manual': False}
]

records = []

for spec in model_specs:
    model_path = models_dir / spec['file']
    model = load_pickle_model(model_path)

    if spec['manual']:
        predictions = predict_manual_linear_regression(model, X_val)
    else:
        predictions = np.asarray(model.predict(X_val)).reshape(-1)

    mae = mean_absolute_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    r2 = r2_score(y_val, predictions)
    within_10pct = np.mean(np.abs(y_val - predictions) <= 0.10 * np.abs(y_val))

    records.append({
        'Model': spec['name'],
        'Family': spec['family'],
        'Model File': spec['file'],
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2,
        'Within 10% Rate': within_10pct,
        'Accuracy (%)': r2 * 100
    })

comparison_df = pd.DataFrame(records)
comparison_df = comparison_df.sort_values(by=['R2 Score', 'RMSE'], ascending=[False, True]).reset_index(drop=True)
comparison_df.insert(0, 'Rank', np.arange(1, len(comparison_df) + 1))
comparison_df['Within 10% Rate (%)'] = comparison_df['Within 10% Rate'] * 100

comparison_export = comparison_df.copy()
comparison_export.to_csv(results_dir / 'model_comparison_summary.csv', index=False)
comparison_export[['Rank', 'Model', 'Family', 'R2 Score', 'RMSE', 'MAE', 'Within 10% Rate (%)']].to_csv(
    results_dir / 'model_comparison_ranking.csv',
    index=False
)

comparison_df

## Styled Comparison Table

In [ ]:
styled_columns = ['Rank', 'Model', 'Family', 'R2 Score', 'Accuracy (%)', 'RMSE', 'MAE', 'Within 10% Rate (%)']

styled_table = (
    comparison_df[styled_columns]
    .style
    .format({
        'R2 Score': '{:.4f}',
        'Accuracy (%)': '{:.2f}',
        'RMSE': '{:.4f}',
        'MAE': '{:.4f}',
        'Within 10% Rate (%)': '{:.2f}'
    })
    .background_gradient(subset=['R2 Score', 'Accuracy (%)', 'Within 10% Rate (%)'], cmap='YlGn')
    .background_gradient(subset=['RMSE', 'MAE'], cmap='YlOrRd_r')
    .set_caption('Validation-set comparison across manual, classical ML, and ensemble regression models')
)

display(styled_table)

best_r2_model = comparison_df.loc[comparison_df['R2 Score'].idxmax(), 'Model']
lowest_rmse_model = comparison_df.loc[comparison_df['RMSE'].idxmin(), 'Model']

print(f'Best model by R2 Score: {best_r2_model}')
print(f'Best model by RMSE: {lowest_rmse_model}')
print('CSV files saved to reports/results/.')

## Visual Dashboard

In [ ]:
plot_df = comparison_df.copy()

fig, axes = plt.subplots(2, 2, figsize=(22, 16), facecolor='#F7F3ED')
fig.suptitle('Ames Housing Model Comparison Dashboard', fontsize=24, fontweight='bold', y=1.02)

sns.barplot(
    data=plot_df,
    x='R2 Score',
    y='Model',
    hue='Family',
    dodge=False,
    palette=family_palette,
    ax=axes[0, 0]
)
axes[0, 0].set_title('Higher Is Better: R2 Score', fontsize=18, fontweight='bold')
axes[0, 0].set_xlabel('R2 Score')
axes[0, 0].set_ylabel('')
axes[0, 0].legend(title='Family', frameon=True)

sns.barplot(
    data=plot_df.sort_values('RMSE', ascending=True),
    x='RMSE',
    y='Model',
    hue='Family',
    dodge=False,
    palette=family_palette,
    ax=axes[0, 1]
)
axes[0, 1].set_title('Lower Is Better: RMSE', fontsize=18, fontweight='bold')
axes[0, 1].set_xlabel('RMSE')
axes[0, 1].set_ylabel('')
axes[0, 1].legend_.remove()

scatter = axes[1, 0].scatter(
    plot_df['RMSE'],
    plot_df['R2 Score'],
    s=plot_df['Within 10% Rate (%)'] * 10,
    c=plot_df['Family'].map(family_palette),
    alpha=0.85,
    edgecolors='black',
    linewidth=1.2
)
for _, row in plot_df.iterrows():
    axes[1, 0].text(row['RMSE'] + 0.001, row['R2 Score'] + 0.001, row['Model'], fontsize=10)
axes[1, 0].set_title('Performance Landscape', fontsize=18, fontweight='bold')
axes[1, 0].set_xlabel('RMSE')
axes[1, 0].set_ylabel('R2 Score')

heatmap_df = plot_df.set_index('Model')[['R2 Score', 'Accuracy (%)', 'RMSE', 'MAE', 'Within 10% Rate (%)']].copy()
heatmap_scaled = heatmap_df.copy()
heatmap_scaled['RMSE'] = 1 - (heatmap_scaled['RMSE'] - heatmap_scaled['RMSE'].min()) / (heatmap_scaled['RMSE'].max() - heatmap_scaled['RMSE'].min() + 1e-9)
heatmap_scaled['MAE'] = 1 - (heatmap_scaled['MAE'] - heatmap_scaled['MAE'].min()) / (heatmap_scaled['MAE'].max() - heatmap_scaled['MAE'].min() + 1e-9)
heatmap_scaled['R2 Score'] = (heatmap_scaled['R2 Score'] - heatmap_scaled['R2 Score'].min()) / (heatmap_scaled['R2 Score'].max() - heatmap_scaled['R2 Score'].min() + 1e-9)
heatmap_scaled['Accuracy (%)'] = (heatmap_scaled['Accuracy (%)'] - heatmap_scaled['Accuracy (%)'].min()) / (heatmap_scaled['Accuracy (%)'].max() - heatmap_scaled['Accuracy (%)'].min() + 1e-9)
heatmap_scaled['Within 10% Rate (%)'] = (heatmap_scaled['Within 10% Rate (%)'] - heatmap_scaled['Within 10% Rate (%)'].min()) / (heatmap_scaled['Within 10% Rate (%)'].max() - heatmap_scaled['Within 10% Rate (%)'].min() + 1e-9)

sns.heatmap(
    heatmap_scaled,
    cmap='YlGnBu',
    linewidths=0.5,
    annot=heatmap_df.round(3),
    fmt='',
    cbar_kws={'label': 'Normalized score'},
    ax=axes[1, 1]
)
axes[1, 1].set_title('Metric Heatmap', fontsize=18, fontweight='bold')
axes[1, 1].set_xlabel('Metrics')
axes[1, 1].set_ylabel('')

for ax in axes.flat:
    ax.set_facecolor('#FFFDF8')

plt.tight_layout()
dashboard_path = figures_dir / 'model_comparison_dashboard.png'
plt.savefig(dashboard_path, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

print(f'Dashboard saved to: {dashboard_path}')

In [ ]:
plt.figure(figsize=(14, 8), facecolor='#F5EFE6')
ax = sns.barplot(
    data=comparison_df,
    x='Model',
    y='Accuracy (%)',
    hue='Family',
    palette=family_palette
)
ax.set_title('Model Accuracy View (R2 Score x 100)', fontsize=20, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Accuracy (%)')
ax.tick_params(axis='x', rotation=25)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10)
plt.tight_layout()
accuracy_chart_path = figures_dir / 'model_accuracy_comparison.png'
plt.savefig(accuracy_chart_path, dpi=300, bbox_inches='tight', facecolor='#F5EFE6')
plt.show()

plt.figure(figsize=(14, 8), facecolor='#F8F7F4')
rmse_ax = sns.barplot(
    data=comparison_df.sort_values('RMSE', ascending=True),
    x='Model',
    y='RMSE',
    hue='Family',
    palette=family_palette
)
rmse_ax.set_title('Validation RMSE Comparison', fontsize=20, fontweight='bold')
rmse_ax.set_xlabel('Model')
rmse_ax.set_ylabel('RMSE')
rmse_ax.tick_params(axis='x', rotation=25)
for container in rmse_ax.containers:
    rmse_ax.bar_label(container, fmt='%.3f', padding=3, fontsize=10)
plt.tight_layout()
rmse_chart_path = figures_dir / 'model_rmse_comparison.png'
plt.savefig(rmse_chart_path, dpi=300, bbox_inches='tight', facecolor='#F8F7F4')
plt.show()

print(f'Accuracy chart saved to: {accuracy_chart_path}')
print(f'RMSE chart saved to: {rmse_chart_path}')

## Final Summary

- The table and charts above compare **all saved models** on the same validation set.
- `R2 Score` is used as the main regression accuracy indicator.
- `RMSE` and `MAE` show error magnitude, while `Within 10% Rate` provides an intuitive tolerance-based score.
- Re-running this notebook will refresh both the report figures and the CSV summaries in the `reports/` directory.
